[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/10_Temporal_Knowledge_Graphs.ipynb)

# 深入探讨:时态知识图谱(Temporal Knowledge Graphs)

## 概述

本 Notebook 使用 Semantica 对**时态知识图谱(TKG)**进行全面深入的探讨。与静态知识图谱不同,TKG 能捕捉事实、关系和实体随时间的演化。这一能力对以下应用至关重要:

- **企业历史分析**:追踪并购和领导层变动。
- **供应链监控**:追溯产品流转与状态变化。
- **金融欺诈检测**:分析交易序列。

我们将构建一个内容丰富的场景,为一个科技生态的历史建模,时间跨度长达 40 年。

### 涵盖的关键组件

1.  **`GraphBuilder`(时态模式)**:构建具备时间感知属性的知识图谱。
2.  **`TemporalGraphQuery`**:执行时间点、时间区间和路径查询。
3.  **`TemporalPatternDetector`**:识别序列和周期性模式。
4.  **`TemporalVersionManager`**:管理快照并比较图的状态。
5.  **`TemporalVisualizer`**:交互式时间线与演化图。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/kg/)

## 安装

In [ ]:
# !pip install semantica[all]

In [ ]:
import json
from datetime import datetime
from semantica.kg import GraphBuilder, TemporalGraphQuery, TemporalPatternDetector, TemporalVersionManager
from semantica.visualization import TemporalVisualizer
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

# Ensure consistent output for reproducibility
import random
random.seed(42)

## 步骤 1:场景定义与数据准备

我们定义一个数据集,描述 "TechCorp" 和 "InnovateInc" 的历史,包括它们的创始人、产品以及最终的合并。

**时间属性**:
- 实体具有 `founded`、`born`、`released` 日期。
- 关系具有 `timestamp`(时间点事件)或 `valid_from`/`valid_to`(时间区间)。

In [ ]:
# 1. Define Entities with Temporal Metadata
entities = [
    # Organizations
    {"id": "org_1", "type": "Organization", "name": "TechCorp", "properties": {"founded": "1980-01-01", "industry": "Hardware"}},
    {"id": "org_2", "type": "Organization", "name": "InnovateInc", "properties": {"founded": "1995-06-15", "industry": "Software"}},
    {"id": "org_3", "type": "Organization", "name": "FutureSystems", "properties": {"founded": "2010-03-10", "industry": "AI"}},
    
    # People
    {"id": "per_1", "type": "Person", "name": "Alice Founder", "properties": {"born": "1955-05-20"}},
    {"id": "per_2", "type": "Person", "name": "Bob Coder", "properties": {"born": "1970-08-12"}},
    {"id": "per_3", "type": "Person", "name": "Charlie CEO", "properties": {"born": "1980-02-28"}},
    
    # Products
    {"id": "prod_1", "type": "Product", "name": "HomePC", "properties": {"released": "1985-11-20"}},
    {"id": "prod_2", "type": "Product", "name": "SoftOS", "properties": {"released": "1998-07-25"}},
    {"id": "prod_3", "type": "Product", "name": "SmartAI", "properties": {"released": "2015-01-10"}}
]

# 2. Define Temporal Relationships
relationships = [
    # Founding Events (Point in time)
    {"source": "per_1", "target": "org_1", "type": "founded", "timestamp": "1980-01-01", "properties": {"timestamp": "1980-01-01"}},
    {"source": "per_2", "target": "org_2", "type": "founded", "timestamp": "1995-06-15", "properties": {"timestamp": "1995-06-15"}},
    
    # Employment (Intervals)
    {"source": "per_1", "target": "org_1", "type": "ceo_of", "valid_from": "1980-01-01", "valid_to": "2000-01-01", "properties": {"role": "CEO"}},
    {"source": "per_3", "target": "org_1", "type": "ceo_of", "valid_from": "2000-01-02", "valid_to": "2023-01-01", "properties": {"role": "CEO"}},
    {"source": "per_2", "target": "org_2", "type": "cto_of", "valid_from": "1995-06-15", "valid_to": "2010-05-01", "properties": {"role": "CTO"}},
    
    # Product Launches
    {"source": "org_1", "target": "prod_1", "type": "launched", "timestamp": "1985-11-20", "properties": {"timestamp": "1985-11-20"}},
    {"source": "org_2", "target": "prod_2", "type": "launched", "timestamp": "1998-07-25", "properties": {"timestamp": "1998-07-25"}},
    {"source": "org_3", "target": "prod_3", "type": "launched", "timestamp": "2015-01-10", "properties": {"timestamp": "2015-01-10"}},
    
    # Corporate Actions
    {"source": "org_1", "target": "org_2", "type": "acquired", "timestamp": "2010-05-01", "properties": {"amount": "$5B", "timestamp": "2010-05-01"}},
    {"source": "org_1", "target": "org_3", "type": "invested_in", "timestamp": "2012-08-15", "properties": {"amount": "$100M", "timestamp": "2012-08-15"}}
]

print(f"Defined {len(entities)} entities and {len(relationships)} temporal relationships.")

## 步骤 2:构建时态图

我们使用 `enable_temporal=True` 的 `GraphBuilder`。这会指示构建器为 `timestamp`、`valid_from` 和 `valid_to` 等时间属性建立索引。

In [ ]:
builder = GraphBuilder(
    enable_temporal=True,
    temporal_granularity="day"  # Can be 'year', 'month', 'day', 'hour'
)

temporal_kg = builder.build(entities, relationships)

# The graph object now contains temporal indices
print("Graph built successfully.")
print(f"Nodes: {len(temporal_kg['entities'])}")
print(f"Edges: {len(temporal_kg['relationships'])}")

## 步骤 3:高级时态查询

我们使用 `TemporalGraphQuery` 提出与时间相关的问题。

In [ ]:
query_engine = TemporalGraphQuery()

# 1. Point-in-Time Query
# "Who was the CEO of TechCorp in 1990?"
ceo_1990 = query_engine.query_at_time(
    temporal_kg,
    query="Find the CEO of TechCorp",
    at_time="1990-06-01"
)
print("CEO in 1990:", [e['id'] for e in ceo_1990.get('entities', [])])

# "Who was the CEO of TechCorp in 2015?"
ceo_2015 = query_engine.query_at_time(
    temporal_kg,
    query="Find the CEO of TechCorp",
    at_time="2015-06-01"
)
print("CEO in 2015:", [e['id'] for e in ceo_2015.get('entities', [])])

# 2. Temporal Path Finding
# "How did Alice (Founder) connect to SmartAI (Product released in 2015)?"
# This requires traversing through time: Alice -> founded TechCorp -> invested in FutureSystems -> launched SmartAI
paths = query_engine.find_temporal_paths(
    graph=temporal_kg,
    source="per_1", # Alice
    target="prod_3", # SmartAI
    start_time="1980-01-01",
    end_time="2020-01-01"
)

print(f"\nFound {len(paths)} temporal paths from Alice to SmartAI.")
for i, path in enumerate(paths):
    print(f"Path {i+1}: {path}")

## 步骤 4:图演化分析

我们可以使用 `analyze_evolution` 分析图的属性如何随时间变化。

In [ ]:
evolution_stats = query_engine.analyze_evolution(
    temporal_kg,
    start_time="1980-01-01",
    end_time="2025-01-01",
    metrics=["count", "diversity", "stability"]
)

print("\nEvolution Statistics (1980-2025):")
print(f"Total Relationships: {evolution_stats.get('count', 'N/A')}")
print(f"Relationship Diversity: {evolution_stats.get('diversity', 'N/A')}")
print(f"Graph Stability: {evolution_stats.get('stability', 'N/A')}")

## 步骤 5:时态模式检测

我们使用 `TemporalPatternDetector` 自动发现重复出现的结构,例如序列(A -> B -> C)或环。

In [ ]:
detector = TemporalPatternDetector()

# Detect sequential patterns (e.g., Founded -> Launched -> Acquired)
sequences = detector.detect_temporal_patterns(
    temporal_kg,
    pattern_type="sequence",
    min_frequency=1
)

print(f"\nDetected {len(sequences)} sequential patterns.")
for seq in sequences[:3]: # Show top 3
    print(f"Pattern: {seq.get('pattern')}")
    print(f"Support: {seq.get('support')}")

## 步骤 6:版本管理与比较

在真实场景中,知识图谱是分批更新的。`TemporalVersionManager` 负责处理这些版本。

In [ ]:
version_manager = TemporalVersionManager()

# Create explicit versions
v1_1990 = version_manager.create_version(temporal_kg, timestamp="1990-01-01", version_label="v1.0 (Early Days)")
v2_2010 = version_manager.create_version(temporal_kg, timestamp="2010-01-01", version_label="v2.0 (Post-Merger)")

# Compare versions
diff = version_manager.compare_versions(v1_1990, v2_2010)

print(f"\nComparing {v1_1990['label']} vs {v2_2010['label']}:")
print(f"New Entities: {diff.get('entities_added', 0)}")
print(f"New Relationships: {diff.get('relationships_added', 0)}")

## 步骤 7:可视化时间线

最后,`TemporalVisualizer` 让数据鲜活起来。我们将创建一条交互式时间线以及一个快照对比图。

In [ ]:
visualizer = TemporalVisualizer()

# 1. Interactive Timeline
# Prepare events for visualization (extract from KG)
def extract_events(graph):
    events = []
    for rel in graph['relationships']:
        # Point events
        if rel.get('timestamp'):
            events.append({
                'timestamp': rel['timestamp'],
                'type': rel['type'],
                'label': f"{rel['source']} -> {rel['target']}",
                'entity': rel['source']
            })
        # Interval events (start)
        if rel.get('valid_from'):
             events.append({
                'timestamp': rel['valid_from'],
                'type': f"{rel['type']} (start)",
                'label': f"{rel['source']} -> {rel['target']}",
                'entity': rel['source']
            })
    return {'events': events}

temporal_data = extract_events(temporal_kg)
timeline_fig = visualizer.visualize_timeline(temporal_data, output="interactive")
# In a notebook, this would render a Plotly figure. 
timeline_fig.show()

# 2. Version History Visualization
history = [
    {"version": "v1.0", "timestamp": "1990-01-01", "changes": "Founding Era"},
    {"version": "v2.0", "timestamp": "2010-01-01", "changes": "Expansion Era"},
    {"version": "v3.0", "timestamp": "2020-01-01", "changes": "AI Era"}
]
history_fig = visualizer.visualize_version_history(history, output="interactive")
history_fig.show()

print("Visualizations generated (render requires Jupyter environment).")

## 总结

在这次深入探讨中,我们:
1.  为复杂的企业历史**建模**,并附带时间元数据。
2.  使用 `GraphBuilder` **构建**了具备时间感知的知识图谱。
3.  **查询**特定的时间切片和区间,以重建历史。
4.  **追踪**时间路径,理解间接关联。
5.  **分析**了图的演化指标。
6.  **管理**版本并可视化时间线。
7.  使用 `TemporalVisualizer` **可视化**数据。

这一工作流构成了 Semantica 中时态智能应用的骨干。